# Chapter 3.6: Worker & ModelRunner — GPU Execution Pipeline

## Learning Objectives

By the end of this notebook, you will:

1. Understand the Worker class and its per-GPU responsibilities
2. Know ModelRunner's role in preparing inputs and running the model
3. Walk through `prepare_model_input()`: how sequences become tensors
4. Walk through `execute_model()`: the actual forward pass pipeline
5. Understand attention metadata construction
6. Know how CUDA graphs integrate with ModelRunner
7. Understand distributed worker communication

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hideak1/vllm_study/blob/main/notebooks/part3/chapter_3.6_worker_model_runner.ipynb)
[![Download Notebook](https://img.shields.io/badge/Download-Notebook-blue)](https://raw.githubusercontent.com/hideak1/vllm_study/main/notebooks/part3/chapter_3.6_worker_model_runner.ipynb)

**How to do the exercises:**
1. **Google Colab (Recommended)**: Click the "Open in Colab" badge above — you get your own copy with free GPU.
2. **Local Jupyter**: Clone the repo, run `./start.sh`, then open notebooks at `http://localhost:8888`.
3. **Exercises**: Look for cells marked with 🏋️ **Exercise** or **Assignment**. Fill in the `TODO` sections and run the cell to check your work.

> **Tip**: In Colab, the notebook is automatically copied to your Drive — your changes are saved there.

## 1. Worker & ModelRunner Architecture

```
Executor
  └── Worker (one per GPU)
      ├── ModelRunner
      │   ├── Model (LlamaForCausalLM, etc.)
      │   ├── Sampler
      │   └── CUDA Graphs (optional)
      └── CacheEngine
          └── GPU KV Cache Tensors
```

**Worker**: Manages one GPU. Handles initialization, model loading, and coordinates execution.

**ModelRunner**: Prepares input tensors from scheduler metadata and runs the model forward pass.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

# ── Figure: Worker Execution Pipeline ──
BLUE = '#4A90D9'; GREEN = '#27AE60'; ORANGE = '#F39C12'; RED = '#E74C3C'; PURPLE = '#8E44AD'

fig, ax = plt.subplots(figsize=(16, 5))
ax.set_xlim(0, 16); ax.set_ylim(0, 5); ax.axis('off')
fig.patch.set_facecolor('white')

steps = [
    (1.5,  2.5, 'Scheduler\nOutput',       ORANGE, 2.2),
    (4.2,  2.5, 'prepare_\nmodel_input()',  BLUE,   2.2),
    (7.2,  2.5, 'execute_\nmodel()',        PURPLE, 2.2),
    (10.2, 2.5, 'Forward\nPass (GPU)',      RED,    2.2),
    (13.2, 2.5, 'Process\nOutputs',         GREEN,  2.2),
]
bh = 1.6

for x, y, label, color, bw in steps:
    rect = mpatches.FancyBboxPatch((x - bw/2, y - bh/2), bw, bh,
        boxstyle="round,pad=0.12", facecolor=color, edgecolor='white',
        linewidth=2, alpha=0.9)
    ax.add_patch(rect)
    ax.text(x, y, label, ha='center', va='center', fontsize=10,
            fontweight='bold', color='white')

# Arrows
for i in range(len(steps) - 1):
    sx = steps[i][0] + steps[i][4]/2
    dx = steps[i+1][0] - steps[i+1][4]/2
    ax.annotate('', xy=(dx, 2.5), xytext=(sx, 2.5),
                arrowprops=dict(arrowstyle='->', color='#2C3E50', lw=2.5))

# Sub-labels
sublabels = [
    (1.5,  0.5, 'seq_group_metadata\nblocks_to_swap'),
    (4.2,  0.5, 'Gather token IDs,\npositions, block tables'),
    (7.2,  0.5, 'Build attn metadata,\nrun model or CUDA graph'),
    (10.2, 0.5, 'Attention + FFN\nper transformer layer'),
    (13.2, 0.5, 'Sample tokens,\ncheck stop criteria'),
]
for x, y, text in sublabels:
    ax.text(x, y, text, ha='center', va='center', fontsize=7.5,
            color='#555', style='italic')

ax.set_title('Worker Execution Pipeline: From Scheduler Output to Sampled Tokens',
             fontsize=14, fontweight='bold', pad=10)
plt.tight_layout(); plt.show()

In [ ]:
# Source: vllm/worker/worker.py (simplified)

worker_code = '''
class Worker(WorkerBase):
    """A worker that executes the model on a single GPU."""
    
    def __init__(
        self,
        model_config: ModelConfig,
        parallel_config: ParallelConfig,
        scheduler_config: SchedulerConfig,
        device_config: DeviceConfig,
        cache_config: CacheConfig,
        local_rank: int,           # GPU index on this node
        rank: int,                 # Global rank across all nodes
        distributed_init_method: str,
        ...
    ):
        self.model_config = model_config
        self.local_rank = local_rank
        self.rank = rank
        
        # These get created during initialization
        self.model_runner: ModelRunner = None
        self.cache_engine: CacheEngine = None
    
    def init_device(self):
        """Initialize the GPU device and distributed environment."""
        # Set CUDA device
        torch.cuda.set_device(self.local_rank)
        
        # Initialize distributed process group (for TP)
        if self.parallel_config.tensor_parallel_size > 1:
            init_distributed_environment(
                world_size=self.parallel_config.tensor_parallel_size,
                rank=self.rank,
                ...
            )
        
        # Create ModelRunner
        self.model_runner = ModelRunner(
            model_config=self.model_config,
            parallel_config=self.parallel_config,
            scheduler_config=self.scheduler_config,
            device_config=self.device_config,
            cache_config=self.cache_config,
            ...
        )
    
    def load_model(self):
        """Load the model weights onto GPU."""
        self.model_runner.load_model()
    
    def determine_num_available_blocks(self) -> Tuple[int, int]:
        """Profile GPU memory to determine KV cache capacity.
        
        This is a CRITICAL step:
        1. Run a dummy forward pass with max possible batch
        2. Measure peak GPU memory usage
        3. Remaining memory = space for KV cache blocks
        """
        self.model_runner.profile_num_available_blocks(
            block_size=self.cache_config.block_size,
            gpu_memory_utilization=self.cache_config.gpu_memory_utilization,
        )
    
    def initialize_cache(self, num_gpu_blocks, num_cpu_blocks):
        """Allocate KV cache tensors."""
        self.cache_engine = CacheEngine(
            cache_config=self.cache_config,
            model_config=self.model_config,
            parallel_config=self.parallel_config,
        )
        # self.cache_engine.gpu_cache is a list of tensors:
        # One per layer, shape: [2, num_blocks, block_size, num_heads, head_dim]
        # The 2 is for K and V
    
    def execute_model(
        self,
        execute_model_req: ExecuteModelRequest,
    ) -> List[SamplerOutput]:
        """Execute the model for one step."""
        
        # Step 1: Perform memory operations (swap in/out, copy)
        self.cache_engine.swap_in(execute_model_req.blocks_to_swap_in)
        self.cache_engine.swap_out(execute_model_req.blocks_to_swap_out)
        self.cache_engine.copy(execute_model_req.blocks_to_copy)
        
        # Step 2: Run the model
        output = self.model_runner.execute_model(
            seq_group_metadata_list=execute_model_req.seq_group_metadata_list,
            kv_caches=self.cache_engine.gpu_cache,
        )
        
        return output
'''
print(worker_code)

## 2. ModelRunner: The GPU Execution Engine

In [ ]:
model_runner_code = '''
class ModelRunner:
    """Responsible for:
    1. Loading the model
    2. Preparing input tensors from scheduler metadata
    3. Running the model forward pass
    4. Sampling next tokens
    """
    
    def __init__(self, model_config, parallel_config, ...):
        self.model_config = model_config
        self.device = torch.device("cuda")
        
        # Model and sampler are loaded later
        self.model = None
        
    def load_model(self):
        """Load model weights from disk/HuggingFace."""
        from vllm.model_executor.model_loader import get_model
        self.model = get_model(
            model_config=self.model_config,
            device_config=self.device_config,
            ...
        )
    
    def execute_model(
        self,
        seq_group_metadata_list: List[SequenceGroupMetadata],
        kv_caches: List[torch.Tensor],
    ) -> List[SamplerOutput]:
        """Execute one forward pass.
        
        This is where sequences become tensors, the model runs,
        and tokens are sampled.
        """
        
        # ╔═══════════════════════════════════╗
        # ║ Step 1: Prepare Model Input       ║
        # ╚═══════════════════════════════════╝
        model_input = self.prepare_model_input(
            seq_group_metadata_list
        )
        # model_input contains:
        #   - input_tokens: [batch_size] token IDs
        #   - input_positions: [batch_size] position indices
        #   - attn_metadata: everything the attention layer needs
        #   - sampling_metadata: everything the sampler needs
        
        # ╔═══════════════════════════════════╗
        # ║ Step 2: Run Model Forward Pass    ║
        # ╚═══════════════════════════════════╝
        # Option A: Use CUDA graph (faster, less flexible)
        # Option B: Run eagerly (slower, more flexible)
        
        if self.cuda_graph_runner and self._can_use_cuda_graph(model_input):
            hidden_states = self.cuda_graph_runner.forward(
                model_input, kv_caches
            )
        else:
            hidden_states = self.model(
                input_ids=model_input.input_tokens,
                positions=model_input.input_positions,
                kv_caches=kv_caches,
                attn_metadata=model_input.attn_metadata,
            )
        # hidden_states: [batch_size, hidden_dim]
        
        # ╔═══════════════════════════════════╗
        # ║ Step 3: Sample Next Tokens        ║
        # ╚═══════════════════════════════════╝
        output = self.model.sample(
            logits=hidden_states,
            sampling_metadata=model_input.sampling_metadata,
        )
        
        return [output]
'''
print(model_runner_code)

## 3. prepare_model_input() — From Sequences to Tensors

In [ ]:
prepare_input_code = '''
def prepare_model_input(
    self,
    seq_group_metadata_list: List[SequenceGroupMetadata],
) -> ModelInput:
    """Prepare all input tensors for the model forward pass.
    
    This is where the abstraction of sequences is converted into
    concrete GPU tensors that the model can consume.
    """
    
    # Separate prefill and decode sequences
    # They have different input characteristics:
    # - Prefill: multiple tokens per sequence (all prompt tokens)
    # - Decode: exactly 1 token per sequence (the last generated token)
    
    input_tokens = []     # Token IDs to process
    input_positions = []  # Position in the sequence for each token
    slot_mapping = []     # Which KV cache slot each token writes to
    
    # For attention metadata
    seq_lens = []         # Length of each sequence
    block_tables = []     # Block table for each sequence
    
    for seq_group_metadata in seq_group_metadata_list:
        is_prompt = seq_group_metadata.is_prompt
        
        for seq_id, seq_data in seq_group_metadata.seq_data.items():
            if is_prompt:
                # PREFILL: include all prompt tokens
                tokens = seq_data.get_token_ids()
                positions = list(range(len(tokens)))
                
                # Compute slot mapping for each token
                # slot = physical_block_number * block_size + offset_in_block
                block_table = seq_group_metadata.block_tables[seq_id]
                for pos in positions:
                    block_idx = pos // self.block_size
                    block_offset = pos % self.block_size
                    physical_block = block_table[block_idx]
                    slot = physical_block * self.block_size + block_offset
                    slot_mapping.append(slot)
            else:
                # DECODE: only the last generated token
                tokens = [seq_data.get_last_token_id()]
                positions = [seq_data.get_len() - 1]
                
                # Slot for the new token
                block_table = seq_group_metadata.block_tables[seq_id]
                pos = seq_data.get_len() - 1
                block_idx = pos // self.block_size
                block_offset = pos % self.block_size
                physical_block = block_table[block_idx]
                slot = physical_block * self.block_size + block_offset
                slot_mapping.append(slot)
            
            input_tokens.extend(tokens)
            input_positions.extend(positions)
            seq_lens.append(seq_data.get_len())
            block_tables.append(
                seq_group_metadata.block_tables[seq_id]
            )
    
    # Convert to tensors
    input_tokens_tensor = torch.tensor(
        input_tokens, dtype=torch.long, device="cuda"
    )
    input_positions_tensor = torch.tensor(
        input_positions, dtype=torch.long, device="cuda"
    )
    slot_mapping_tensor = torch.tensor(
        slot_mapping, dtype=torch.long, device="cuda"
    )
    
    # Build attention metadata
    attn_metadata = self._build_attn_metadata(
        seq_lens, block_tables, slot_mapping_tensor, ...
    )
    
    # Build sampling metadata
    sampling_metadata = self._build_sampling_metadata(
        seq_group_metadata_list
    )
    
    return ModelInput(
        input_tokens=input_tokens_tensor,
        input_positions=input_positions_tensor,
        attn_metadata=attn_metadata,
        sampling_metadata=sampling_metadata,
    )
'''
print(prepare_input_code)

In [ ]:
# Concrete example of input preparation

example = """
Example: Mixed Prefill + Decode Batch
══════════════════════════════════════

Scheduled sequences:
  Seq A (PREFILL): prompt="Hello world" → tokens [15496, 995], block_table=[P3]
  Seq B (DECODE): prompt+gen 10 tokens total, last token=512, block_table=[P7, P1]
  Seq C (DECODE): prompt+gen 5 tokens total, last token=256, block_table=[P5]

block_size = 8

After prepare_model_input():

  input_tokens:    [15496, 995,  512,  256]
                    ↑prefill A↑  ↑B↑   ↑C↑
  
  input_positions: [0,     1,    9,    4]
                    ↑pos 0↑ ↑1↑  ↑9↑  ↑4↑
  
  slot_mapping:    [P3*8+0, P3*8+1, P1*8+1, P5*8+4]
                   = [24,    25,     9,      44]
  
  seq_lens:        [2, 10, 5]
  
  block_tables:
    Seq A: [3]
    Seq B: [7, 1]
    Seq C: [5]

The slot_mapping tells the attention kernel EXACTLY where to write
each token's KV cache in the pre-allocated cache tensor.
"""
print(example)

## 4. Attention Metadata

In [ ]:
attn_metadata_code = '''
class AttentionMetadata:
    """Metadata for the attention layer.
    
    Different attention backends need different metadata.
    This is the common base; each backend extends it.
    """
    
    # Prefill metadata
    num_prefills: int              # Number of prefill sequences
    num_prefill_tokens: int        # Total prefill tokens
    
    # Decode metadata  
    num_decode_tokens: int         # Total decode tokens (= num decode seqs)
    
    # Block tables for PagedAttention
    # Shape: [num_seqs, max_num_blocks_per_seq]
    block_tables: torch.Tensor
    
    # Sequence lengths
    seq_lens: List[int]            # Length of each sequence
    seq_lens_tensor: torch.Tensor  # Same, as GPU tensor
    
    # Slot mapping (where to write KV cache)
    slot_mapping: torch.Tensor     # [total_tokens]
    
    # For mixed prefill + decode batches:
    # Need to know which tokens are prefill vs decode
    # Different attention patterns:
    #   - Prefill: causal self-attention on prompt
    #   - Decode: attention to ALL previous tokens via KV cache


# FlashAttention-specific metadata adds:
class FlashAttentionMetadata(AttentionMetadata):
    # For variable-length sequences in FlashAttention
    max_prefill_seq_len: int       # Max prefill sequence length
    max_decode_seq_len: int        # Max decode context length
    
    # Query start indices for each sequence
    query_start_loc: torch.Tensor  # [num_seqs + 1]
    seq_start_loc: torch.Tensor    # [num_seqs + 1]
    
    # For FlashAttention's variable-length API
    # These describe where each sequence's data starts
    # in the packed input tensor
'''
print(attn_metadata_code)

## 5. CUDA Graph Integration

In [ ]:
cuda_graph_info = """
CUDA Graphs in ModelRunner
══════════════════════════

What are CUDA Graphs?
  CUDA graphs capture a sequence of GPU operations (kernels, memory ops)
  into a single graph that can be replayed without CPU involvement.
  This eliminates kernel launch overhead.

Why does vLLM use them?
  In the decode phase, each step processes exactly 1 token per sequence.
  The computation graph is the SAME for the same batch size.
  Without CUDA graphs: CPU launches ~100 kernels per step (overhead ~1ms).
  With CUDA graphs: CPU replays 1 graph per step (overhead ~0.01ms).
  For short decode steps, this is a 10-50% speedup!

How vLLM captures CUDA graphs:
  1. At startup, capture graphs for common batch sizes: 1, 2, 4, 8, ..., 256
  2. Pre-allocate fixed-size input buffers for each batch size
  3. During execution:
     a. Find the smallest captured graph >= current batch size
     b. Copy actual inputs into the pre-allocated buffers
     c. Replay the graph
     d. Read outputs from pre-allocated output buffers

Limitations:
  - Only for DECODE phase (fixed computation per token)
  - Cannot use for PREFILL (variable number of tokens)
  - Batch size must match a pre-captured size (pad if needed)
  - Cannot use with enforce_eager=True
  - Makes debugging harder (can't breakpoint inside graph)

Code flow:
  ModelRunner.__init__():
    self.graph_runners = {}  # batch_size → CUDAGraphRunner
  
  ModelRunner.capture_model():
    for batch_size in [1, 2, 4, 8, ..., max_batch_size]:
      # Create dummy inputs
      # Warm up (run once without capture)
      # Capture the graph
      graph = torch.cuda.CUDAGraph()
      with torch.cuda.graph(graph):
          output = self.model(dummy_input, ...)
      self.graph_runners[batch_size] = CUDAGraphRunner(graph, ...)
  
  ModelRunner.execute_model():
    if can_use_graph:
      # Find appropriate graph runner
      runner = self.graph_runners[padded_batch_size]
      # Copy real inputs into graph's input buffers
      runner.copy_inputs(real_input)
      # Replay
      runner.forward()
      # Read outputs
      return runner.get_outputs()
"""
print(cuda_graph_info)

## 6. CacheEngine — GPU KV Cache Management

In [ ]:
cache_engine_code = '''
class CacheEngine:
    """Manages the physical KV cache tensors on GPU and CPU."""
    
    def __init__(self, cache_config, model_config, parallel_config):
        # Calculate KV cache shape per layer
        self.num_layers = model_config.get_num_layers(parallel_config)
        self.num_kv_heads = model_config.get_num_kv_heads(parallel_config)
        self.head_dim = model_config.get_head_size()
        self.block_size = cache_config.block_size
        
        # Allocate GPU KV cache
        # Shape per layer: [2, num_blocks, block_size, num_kv_heads, head_dim]
        #   2 = Key + Value
        self.gpu_cache: List[torch.Tensor] = self._allocate_kv_cache(
            num_blocks=cache_config.num_gpu_blocks,
            device="cuda",
        )
        # gpu_cache[layer_idx] is one tensor per layer
        
        # Allocate CPU KV cache (for swap space)
        self.cpu_cache: List[torch.Tensor] = self._allocate_kv_cache(
            num_blocks=cache_config.num_cpu_blocks,
            device="cpu",
        )
    
    def _allocate_kv_cache(
        self, num_blocks: int, device: str
    ) -> List[torch.Tensor]:
        """Allocate KV cache tensors."""
        kv_cache: List[torch.Tensor] = []
        for _ in range(self.num_layers):
            # Key cache and Value cache
            key_cache = torch.empty(
                num_blocks, self.block_size,
                self.num_kv_heads, self.head_dim,
                dtype=self.dtype, device=device,
            )
            value_cache = torch.empty(
                num_blocks, self.block_size,
                self.num_kv_heads, self.head_dim,
                dtype=self.dtype, device=device,
            )
            kv_cache.append((key_cache, value_cache))
        return kv_cache
    
    def swap_in(self, src_to_dst: Dict[int, int]):
        """Copy KV cache from CPU to GPU."""
        for src_block, dst_block in src_to_dst.items():
            for layer_idx in range(self.num_layers):
                # Copy key cache
                self.gpu_cache[layer_idx][0][dst_block].copy_(
                    self.cpu_cache[layer_idx][0][src_block]
                )
                # Copy value cache
                self.gpu_cache[layer_idx][1][dst_block].copy_(
                    self.cpu_cache[layer_idx][1][src_block]
                )
    
    def swap_out(self, src_to_dst: Dict[int, int]):
        """Copy KV cache from GPU to CPU."""
        # Similar to swap_in but reversed direction
        ...
    
    def copy(self, src_to_dsts: Dict[int, int]):
        """Copy blocks within GPU (for Copy-on-Write)."""
        # Uses a custom CUDA kernel for efficient block copying
        cache_ops.copy_blocks(
            self.gpu_cache, src_to_dsts
        )
'''
print(cache_engine_code)

## 7. Forward Pass Visualization

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

fig, ax = plt.subplots(1, 1, figsize=(16, 10))
ax.set_xlim(0, 16)
ax.set_ylim(0, 11)
ax.axis('off')
ax.set_title('ModelRunner Forward Pass Pipeline', fontsize=16, fontweight='bold')

# Stages
stages = [
    (8, 10, 'SequenceGroupMetadata\n(from Scheduler)', '#FFE0B2', 12),
    (8, 8.5, 'prepare_model_input()\ntokens, positions, attn_metadata', '#C8E6C9', 10),
    (8, 7, 'Embedding Layer\ntoken_ids → hidden_states', '#B3E5FC', 10),
    (8, 5.5, 'Transformer Layers (x N)\nAttention + FFN', '#E1BEE7', 10),
    (4, 4, 'PagedAttention\nRead/Write KV Cache', '#FFCDD2', 6),
    (12, 4, 'FFN (MLP)\nSiLU + Linear', '#F0F4C3', 6),
    (8, 2.5, 'LM Head\nhidden → logits [vocab_size]', '#B2DFDB', 10),
    (8, 1, 'Sampler\nlogits → next token', '#FFE0B2', 10),
]

for x, y, text, color, width in stages:
    w = width * 0.13
    rect = mpatches.FancyBboxPatch((x-w, y-0.5), 2*w, 1.0,
                                    boxstyle="round,pad=0.1",
                                    facecolor=color, edgecolor='black', linewidth=1.5)
    ax.add_patch(rect)
    ax.text(x, y, text, ha='center', va='center', fontsize=8)

# Main arrows
for i in range(len(stages)-1):
    if i == 3:  # Split to attention + FFN
        ax.annotate('', xy=(4, 4.5), xytext=(8, 5),
                    arrowprops=dict(arrowstyle='->', lw=1.5))
        ax.annotate('', xy=(12, 4.5), xytext=(8, 5),
                    arrowprops=dict(arrowstyle='->', lw=1.5))
    elif i == 4:  # Attention
        ax.annotate('', xy=(8, 3), xytext=(4, 3.5),
                    arrowprops=dict(arrowstyle='->', lw=1.5))
    elif i == 5:  # FFN
        ax.annotate('', xy=(8, 3), xytext=(12, 3.5),
                    arrowprops=dict(arrowstyle='->', lw=1.5))
    elif i > 5:
        ax.annotate('', xy=(8, stages[i+1][1]+0.5), xytext=(8, stages[i][1]-0.5),
                    arrowprops=dict(arrowstyle='->', lw=1.5))
    elif i < 3:
        ax.annotate('', xy=(8, stages[i+1][1]+0.5), xytext=(8, stages[i][1]-0.5),
                    arrowprops=dict(arrowstyle='->', lw=1.5))

# KV Cache annotation
ax.text(1, 4, 'KV Cache\n(GPU blocks)', ha='center', fontsize=8,
        bbox=dict(boxstyle='round', facecolor='lightyellow', edgecolor='orange'))
ax.annotate('', xy=(2.7, 4), xytext=(2, 4),
            arrowprops=dict(arrowstyle='<->', color='orange', lw=1.5))

plt.tight_layout()
plt.show()

## 8. Distributed Worker Communication

In [ ]:
distributed_info = """
Distributed Worker Communication (Tensor Parallelism)
═══════════════════════════════════════════════════════

When using tensor parallelism (TP), the model is split across GPUs.
Workers need to communicate during the forward pass.

Communication Pattern:
  ┌─────────┐  ┌─────────┐  ┌─────────┐  ┌─────────┐
  │ Worker 0 │  │ Worker 1 │  │ Worker 2 │  │ Worker 3 │
  │ GPU 0    │  │ GPU 1    │  │ GPU 2    │  │ GPU 3    │
  └────┬─────┘  └────┬─────┘  └────┬─────┘  └────┬─────┘
       │              │              │              │
       └──────── AllReduce ──────────┘              │
       └───────────── AllReduce ────────────────────┘

In each transformer layer:
  1. Column-parallel Linear (QKV projection):
     - Each GPU has 1/TP of the columns
     - No communication needed (embarrassingly parallel)
  
  2. Self-Attention:
     - Each GPU computes attention for its heads
     - No communication (each head is independent)
  
  3. Row-parallel Linear (output projection):
     - Each GPU has 1/TP of the rows
     - AllReduce to combine partial results
  
  4. MLP:
     - Similar to above: column-parallel + row-parallel
     - One AllReduce per MLP

Total AllReduces per layer: 2 (one for attention, one for MLP)
Total AllReduces per step: 2 * num_layers

Communication: NCCL (NVIDIA Collective Communications Library)
Bandwidth: NVLink provides 600 GB/s between GPUs on the same node
"""
print(distributed_info)

## 9. Profiling the Forward Pass

In [ ]:
profiling_code = '''
# How to profile a forward pass (requires GPU)

from vllm import LLM, SamplingParams
import torch

# Create LLM with profiling-friendly settings
llm = LLM(
    model="facebook/opt-125m",
    enforce_eager=True,  # No CUDA graphs (easier to profile)
    gpu_memory_utilization=0.5,
)

# Profile with torch.profiler
with torch.profiler.profile(
    activities=[
        torch.profiler.ProfilerActivity.CPU,
        torch.profiler.ProfilerActivity.CUDA,
    ],
    record_shapes=True,
    with_stack=True,
) as prof:
    outputs = llm.generate(
        ["Hello, world!"],
        SamplingParams(max_tokens=10),
    )

# Print profiling results
print(prof.key_averages().table(
    sort_by="cuda_time_total", row_limit=20
))

# Expected output shows:
# - Attention kernels (flash_attn or paged_attn)
# - Linear layers (GEMM operations)
# - Activation functions
# - Sampling operations
# - Memory operations (if any swaps)
'''
print(profiling_code)

In [ ]:
# Typical profiling breakdown

import matplotlib.pyplot as plt

# Approximate breakdown for a 7B model decode step
components = ['Attention\nKernels', 'Linear\n(GEMM)', 'Activation\n(SiLU/GELU)', 
              'LayerNorm', 'Sampling', 'Other\n(overhead)']
times_ms = [0.8, 3.5, 0.3, 0.2, 0.1, 0.1]
colors = ['#FF6B6B', '#4ECDC4', '#45B7D1', '#96CEB4', '#FFEAA7', '#DDA0DD']

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Bar chart
bars = ax1.bar(components, times_ms, color=colors, edgecolor='black')
ax1.set_ylabel('Time (ms)')
ax1.set_title('Decode Step Breakdown (~7B model, batch=32)', fontweight='bold')
for bar, t in zip(bars, times_ms):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.05,
             f'{t}ms', ha='center', fontsize=9)

# Pie chart
ax2.pie(times_ms, labels=components, colors=colors, autopct='%1.0f%%',
        startangle=90)
ax2.set_title('Time Distribution', fontweight='bold')

plt.tight_layout()
plt.show()

total = sum(times_ms)
print(f"Total decode step time: {total:.1f}ms")
print(f"Throughput at batch=32: {32/total*1000:.0f} tokens/s")
print(f"\nKey insight: Linear layers (GEMM) dominate the decode step.")
print(f"Attention becomes more expensive with longer sequences.")

## 10. Memory Profiling

In [ ]:
memory_profiling = '''
Worker.determine_num_available_blocks() — Detailed
════════════════════════════════════════════════════

def determine_num_available_blocks(self) -> Tuple[int, int]:
    """Profile to determine how many KV cache blocks fit.
    
    Strategy:
    1. Measure current GPU memory usage (model weights loaded)
    2. Run a dummy forward pass with max batch size
    3. Measure peak GPU memory (weights + activations)
    4. Remaining = total_gpu_memory * utilization - peak_memory
    5. num_blocks = remaining / bytes_per_block
    """
    
    # Step 1: Free any CUDA cache from model loading
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()
    
    # Step 2: Run profiling forward pass
    # This uses a dummy batch with max_num_batched_tokens tokens
    self.model_runner.profile_run()
    
    # Step 3: Measure peak memory
    torch.cuda.synchronize()
    peak_memory = torch.cuda.max_memory_allocated()
    
    # Step 4: Calculate available memory
    total_gpu_memory = torch.cuda.get_device_properties(
        self.local_rank
    ).total_memory
    
    available = total_gpu_memory * self.cache_config.gpu_memory_utilization
    cache_memory = available - peak_memory
    
    # Step 5: Calculate number of blocks
    cache_block_size = CacheEngine.get_cache_block_size(
        self.cache_config, self.model_config, self.parallel_config
    )
    # cache_block_size = 2 * num_layers * block_size * num_kv_heads * head_dim * dtype_bytes
    
    num_gpu_blocks = int(cache_memory // cache_block_size)
    
    # CPU blocks
    num_cpu_blocks = int(self.cache_config.swap_space_bytes // cache_block_size)
    
    return num_gpu_blocks, num_cpu_blocks


# Example calculation for LLaMA-7B on A100 80GB:
#   Total GPU memory:     80 GB
#   gpu_memory_utilization: 0.90
#   Available:            72 GB
#   Model weights:        ~14 GB (FP16)
#   Peak activations:     ~2 GB (varies with batch size)
#   Cache memory:         72 - 14 - 2 = 56 GB
#   Block size:           16 tokens
#   Bytes per block:      2 * 32 * 16 * 32 * 128 * 2 = 8,388,608 bytes = 8 MB
#   Number of blocks:     56 GB / 8 MB = 7,168 blocks
#   Total cached tokens:  7,168 * 16 = 114,688 tokens
'''
print(memory_profiling)

## Exercises

### Exercise 1: Calculate Batch Composition

Given a batch with 3 prefill sequences (lengths 128, 256, 64) and 5 decode sequences, calculate what the input tensors look like.

In [ ]:
# Exercise 1 Solution

print("Batch Composition Exercise")
print("=" * 50)

prefill_lens = [128, 256, 64]
decode_seqs = 5

total_prefill_tokens = sum(prefill_lens)
total_decode_tokens = decode_seqs  # 1 token each
total_tokens = total_prefill_tokens + total_decode_tokens

print(f"Prefill sequences: {len(prefill_lens)}")
print(f"  Lengths: {prefill_lens}")
print(f"  Total prefill tokens: {total_prefill_tokens}")
print(f"")
print(f"Decode sequences: {decode_seqs}")
print(f"  Total decode tokens: {total_decode_tokens}")
print(f"")
print(f"Total tokens in batch: {total_tokens}")
print(f"")
print(f"input_tokens shape: [{total_tokens}]")
print(f"  [{', '.join(['t']*3)}...{total_prefill_tokens} prefill tokens..., "
      f"{', '.join(['d']*decode_seqs)} decode tokens]")
print(f"")
print(f"input_positions shape: [{total_tokens}]")
print(f"  Prefill seq 0: [0, 1, 2, ..., 127]")
print(f"  Prefill seq 1: [0, 1, 2, ..., 255]")
print(f"  Prefill seq 2: [0, 1, 2, ..., 63]")
print(f"  Decode tokens: [pos_d0, pos_d1, pos_d2, pos_d3, pos_d4]")
print(f"  (decode positions are each sequence's current length - 1)")
print(f"")
print(f"attn_metadata:")
print(f"  num_prefills: {len(prefill_lens)}")
print(f"  num_prefill_tokens: {total_prefill_tokens}")
print(f"  num_decode_tokens: {total_decode_tokens}")
print(f"  seq_lens: {prefill_lens} + [len_d0, len_d1, len_d2, len_d3, len_d4]")
print(f"  block_tables: one per sequence ({len(prefill_lens) + decode_seqs} total)")

### Exercise 2: CUDA Graph Batch Size Selection

In [ ]:
# Exercise 2: Implement CUDA graph batch size selection

def select_cuda_graph_batch_size(actual_batch_size, captured_sizes):
    """Find the smallest captured CUDA graph size >= actual batch size."""
    for size in sorted(captured_sizes):
        if size >= actual_batch_size:
            return size
    return None  # Too large, must run eagerly

# Standard captured sizes
captured = [1, 2, 4, 8, 16, 32, 64, 128, 256]

test_cases = [1, 3, 5, 15, 33, 100, 200, 300]
print(f"Captured CUDA graph sizes: {captured}")
print(f"{'Actual':<10} {'Selected':<10} {'Padding':<10} {'Waste %':<10}")
print("=" * 40)
for actual in test_cases:
    selected = select_cuda_graph_batch_size(actual, captured)
    if selected:
        padding = selected - actual
        waste = padding / selected * 100
        print(f"{actual:<10} {selected:<10} {padding:<10} {waste:<10.1f}")
    else:
        print(f"{actual:<10} {'EAGER':<10} {'N/A':<10} {'N/A':<10}")

print("\nNote: Padding waste means we run extra computation for dummy tokens.")
print("The CUDA graph overhead savings usually outweigh the padding waste.")

## Summary

Key takeaways:

1. **Worker** manages one GPU: initialization, model loading, cache management, and execution coordination
2. **ModelRunner** converts scheduler metadata into GPU tensors and runs the forward pass
3. **prepare_model_input()** transforms sequence metadata into input_tokens, positions, attention metadata, and sampling metadata
4. **execute_model()** runs the actual forward pass: embedding → transformer layers → LM head → sampling
5. **CacheEngine** manages physical KV cache tensors and swap operations
6. **CUDA graphs** eliminate kernel launch overhead for decode steps
7. **Tensor parallelism** splits the model across GPUs with AllReduce communication

**Next**: Chapter 3.7 explores Model Loading and Weight Management.